In [3]:
Libraries import

SyntaxError: invalid syntax (3357831262.py, line 1)

In [ ]:
import pandas as pd


In [ ]:
import os
os.getcwd()


In [ ]:
os.listdir()


In [ ]:
import os
os.getcwd()

In [ ]:
import os
os.listdir("Smarrt_inventory_project")

In [ ]:
df = pd.read_csv("Smarrt_inventory_project/demand_forecasting.csv")
df.head()

In [4]:
DATASET SIZE:

SyntaxError: invalid syntax (2481903284.py, line 1)

In [ ]:
df.shape

In [ ]:
COLUMN NAMES CHECK:

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
MISSING VALUES CHECK:

In [ ]:
df.isnull().sum()

In [ ]:
DUPLICATE ROWS CHECK:

In [5]:
df.duplicated().sum()

NameError: name 'df' is not defined

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.duplicated().sum()

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

In [ ]:
df["Date"].dtype

In [6]:
df.describe()

NameError: name 'df' is not defined

In [ ]:
print("Negative Inventory:", (df["Inventory Level"] < 0).sum())
print("Negative Units Sold:", (df["Units Sold"] < 0).sum())
print("Negative Units Ordered:", (df["Units Ordered"] < 0).sum())
print("Negative Price:", (df["Price"] < 0).sum())

In [ ]:
import numpy as np

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
print(df.columns.tolist())

In [ ]:
print(df.dtypes)

In [ ]:
categorical_cols = df.select_dtypes(include="str").columns

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna(subset=["Date"])

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.to_csv("cleaned_inventory_data.csv", index=False)

In [ ]:
import os

print(os.getcwd())
print(os.listdir())

In [ ]:
Category-wise analysis

In [7]:
df["Category"].value_counts()

NameError: name 'df' is not defined

In [ ]:
df["Region"].value_counts()

In [ ]:
df["Weather Condition"].value_counts()

In [ ]:
df["Seasonality"].value_counts()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("cleaned_inventory_data.csv")

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df["Demand"].describe()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(df["Demand"], bins=30)

plt.xlabel("Demand")
plt.ylabel("Frequency")
plt.title("Demand Distribution")

plt.show()

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])

In [ ]:
daily_demand = df.groupby("Date")["Demand"].sum().reset_index()

daily_demand.head()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(daily_demand["Date"], daily_demand["Demand"])

plt.xlabel("Date")
plt.ylabel("Total Demand")
plt.title("Demand Trend Over Time")

plt.show()

In [8]:
FEATURE ENGINEERING Date features

SyntaxError: invalid syntax (3621505788.py, line 1)

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])

df["Day"] = df["Date"].dt.day
df["Month"] = df["Date"].dt.month
df["DayOfWeek"] = df["Date"].dt.dayofweek
df["Year"] = df["Date"].dt.year

In [ ]:
Product-wise previous demand

In [ ]:
df = df.sort_values(["Product ID", "Date"])

df["Lag_1"] = df.groupby("Product ID")["Demand"].shift(1)

df["Lag_7"] = df.groupby("Product ID")["Demand"].shift(7)

In [ ]:
3 — 7-day rolling average

In [ ]:
df["Rolling_Mean_7"] = (
    df.groupby("Product ID")["Demand"]
      .transform(lambda x: x.shift(1).rolling(7).mean())
)

In [ ]:
df[["Demand", "Lag_1", "Lag_7", "Rolling_Mean_7"]].head(15)

In [ ]:
df[["Lag_1", "Lag_7", "Rolling_Mean_7"]].isnull().sum()

In [ ]:
df = df.dropna(subset=["Lag_1", "Lag_7", "Rolling_Mean_7"])

In [ ]:
Final features selection

In [9]:
features = [
    "Day",
    "Month",
    "DayOfWeek",
    "Year",
    "Lag_1",
    "Lag_7",
    "Rolling_Mean_7",
    "Inventory Level",
    "Units Ordered",
    "Price",
    "Discount",
    "Promotion",
    "Competitor Pricing",
    "Epidemic"
]

X = df[features]
y = df["Demand"]

NameError: name 'df' is not defined

In [ ]:
Train-Test Split

In [ ]:
df = df.sort_values("Date").reset_index(drop=True)

In [ ]:
80% Train, 20% Test

In [ ]:
split = int(len(df) * 0.8)

train = df.iloc[:split]
test = df.iloc[split:]

print("Training data:", len(train))
print("Testing data:", len(test))

In [ ]:
X_train = train[features]
y_train = train["Demand"]

X_test = test[features]
y_test = test["Demand"]

In [ ]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

In [10]:
Random Forest Model Training

SyntaxError: invalid syntax (2584780538.py, line 1)

In [ ]:
%pip install scikit-learn


In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
print(y_pred[:10])

In [ ]:
comparison = pd.DataFrame({
    "Actual Demand": y_test.values,
    "Predicted Demand": y_pred
})

comparison.head(10)

In [11]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_pred)

print("Mean Absolute Error:", mae)

NameError: name 'y_test' is not defined

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Root Mean Squared Error:", rmse)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(y_test.values, label="Actual Demand")
plt.plot(y_pred, label="Predicted Demand")

plt.xlabel("Test Data")
plt.ylabel("Demand")
plt.title("Actual vs Predicted Demand")

plt.legend()
plt.show()

In [ ]:
Demand Forecasting basic model completed

In [ ]:
def forecast_future_demand(model, df, product_id, days=7):

    product_data = df[df["Product ID"] == product_id].copy()
    product_data = product_data.sort_values("Date")

    history = list(product_data["Demand"].values)

    last_date = product_data["Date"].max()

    future_predictions = []

    for i in range(days):

        future_date = last_date + pd.Timedelta(days=i + 1)

        lag_1 = history[-1]
        lag_7 = history[-7] if len(history) >= 7 else history[0]
        rolling_mean_7 = np.mean(history[-7:])

        features_future = pd.DataFrame({
            "Day": [future_date.day],
            "Month": [future_date.month],
            "DayOfWeek": [future_date.dayofweek],
            "Year": [future_date.year],
            "Lag_1": [lag_1],
            "Lag_7": [lag_7],
            "Rolling_Mean_7": [rolling_mean_7],
            "Inventory Level": [product_data["Inventory Level"].iloc[-1]],
            "Units Ordered": [product_data["Units Ordered"].iloc[-1]],
            "Price": [product_data["Price"].iloc[-1]],
            "Discount": [product_data["Discount"].iloc[-1]],
            "Promotion": [product_data["Promotion"].iloc[-1]],
            "Competitor Pricing": [product_data["Competitor Pricing"].iloc[-1]],
            "Epidemic": [product_data["Epidemic"].iloc[-1]]
        })

        prediction = model.predict(features_future)[0]

        future_predictions.append({
            "Date": future_date,
            "Predicted Demand": prediction
        })

        history.append(prediction)

    return pd.DataFrame(future_predictions)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("cleaned_inventory_data.csv")

In [ ]:
df["Product ID"].unique()[:10]

In [ ]:
def forecast_future_demand(model, df, product_id, days=7):

    product_data = df[df["Product ID"] == product_id].copy()
    product_data = product_data.sort_values("Date")

    history = list(product_data["Demand"].values)

    last_date = product_data["Date"].max()

    future_predictions = []

    for i in range(days):

        future_date = last_date + pd.Timedelta(days=i + 1)

        lag_1 = history[-1]
        lag_7 = history[-7] if len(history) >= 7 else history[0]
        rolling_mean_7 = np.mean(history[-7:])

        features_future = pd.DataFrame({
            "Day": [future_date.day],
            "Month": [future_date.month],
            "DayOfWeek": [future_date.dayofweek],
            "Year": [future_date.year],
            "Lag_1": [lag_1],
            "Lag_7": [lag_7],
            "Rolling_Mean_7": [rolling_mean_7],
            "Inventory Level": [product_data["Inventory Level"].iloc[-1]],
            "Units Ordered": [product_data["Units Ordered"].iloc[-1]],
            "Price": [product_data["Price"].iloc[-1]],
            "Discount": [product_data["Discount"].iloc[-1]],
            "Promotion": [product_data["Promotion"].iloc[-1]],
            "Competitor Pricing": [product_data["Competitor Pricing"].iloc[-1]],
            "Epidemic": [product_data["Epidemic"].iloc[-1]]
        })

        prediction = model.predict(features_future)[0]

        future_predictions.append({
            "Date": future_date,
            "Predicted Demand": prediction
        })

        history.append(prediction)

    return pd.DataFrame(future_predictions)

In [ ]:
df["Product ID"].unique()[:20]

In [ ]:
"P001" in df["Product ID"].unique()

In [ ]:
product_id = df["Product ID"].iloc[0]

print("Selected Product:", product_id)

In [ ]:
df.groupby("Product ID")["Demand"].count().sort_values(ascending=False).head(10)

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])

df["Day"] = df["Date"].dt.day
df["Month"] = df["Date"].dt.month
df["DayOfWeek"] = df["Date"].dt.dayofweek
df["Year"] = df["Date"].dt.year

In [ ]:
df = df.sort_values(["Product ID", "Date"])

df["Lag_1"] = df.groupby("Product ID")["Demand"].shift(1)
df["Lag_7"] = df.groupby("Product ID")["Demand"].shift(7)

df["Rolling_Mean_7"] = (
    df.groupby("Product ID")["Demand"]
      .transform(lambda x: x.shift(1).rolling(7).mean())
)

df = df.dropna(
    subset=["Lag_1", "Lag_7", "Rolling_Mean_7"]
)

In [ ]:
features = [
    "Day", "Month", "DayOfWeek", "Year",
    "Lag_1", "Lag_7", "Rolling_Mean_7",
    "Inventory Level", "Units Ordered", "Price",
    "Discount", "Promotion", "Competitor Pricing", "Epidemic"
]

df = df.sort_values("Date").reset_index(drop=True)

split = int(len(df) * 0.8)

train = df.iloc[:split]
test = df.iloc[split:]

X_train = train[features]
y_train = train["Demand"]

X_test = test[features]
y_test = test["Demand"]

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Model fitted successfully!")

In [ ]:
future = forecast_future_demand(
    model,
    df,
    "P0001",
    days=7
)

future

In [ ]:
future

In [ ]:
total_forecast = future["Predicted Demand"].sum()

print("Expected demand for next 7 days:", round(total_forecast, 2))

In [ ]:
total_forecast = future["Predicted Demand"].sum()

In [ ]:
print("Total Forecast:", round(total_forecast, 2))

In [ ]:
total_forecast = future["Predicted Demand"].sum()

with open("forecast.txt", "w") as f:
    f.write(str(round(total_forecast, 2)))

In [ ]:
import joblib

joblib.dump(model, "demand_model.pkl")

print("Model saved successfully!")

In [ ]:

import pandas as pd
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor

BASE_DIR = Path(r"C:\Users\nagam\Desktop\Smarrt_inventory_project")

# Load data
df = pd.read_csv(BASE_DIR / "cleaned_inventory_data.csv")
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date"])
df = df.sort_values(["Product ID", "Date"]).reset_index(drop=True)

# Create features
df["Day"] = df["Date"].dt.day
df["Month"] = df["Date"].dt.month
df["DayOfWeek"] = df["Date"].dt.dayofweek
df["Year"] = df["Date"].dt.year

df["Lag_1"] = df.groupby("Product ID")["Demand"].shift(1)
df["Lag_7"] = df.groupby("Product ID")["Demand"].shift(7)

df["Rolling_Mean_7"] = (
    df.groupby("Product ID")["Demand"]
      .transform(lambda x: x.shift(1).rolling(7).mean())
)

features = [
    "Day", "Month", "DayOfWeek", "Year",
    "Lag_1", "Lag_7", "Rolling_Mean_7",
    "Inventory Level", "Units Ordered", "Price",
    "Discount", "Promotion", "Competitor Pricing", "Epidemic"
]

df = df.dropna(subset=features + ["Demand"])

X = df[features]
y = df["Demand"]

# Time-ordered train-test split
split = int(len(df) * 0.8)
X_train, y_train = X.iloc[:split], y.iloc[:split]

# Train
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# Verify before saving
print("Fitted estimators:", len(model.estimators_))

# Save trained model
model_path = BASE_DIR / "demand_model.pkl"
joblib.dump(model, model_path)

# Verify saved file
saved_model = joblib.load(model_path)
print("Saved model estimators:", len(saved_model.estimators_))
print("Trained model saved and verified!")

In [ ]:
df.columns.tolist()